# Titanic + Kaggle Notebook Scratch

### TODOs

* Probability calibration
* Feature engineering
* Other models: Decision trees, Linear, KNN, NN
    * How would we deal with something like a tabnet model in the stacking?
* 2nd layer of stacking
* Other types of top-layer models
* Pseudo-labels

In [1]:
# Imports
import os
import pickle
import time

from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, 
    StackingClassifier
)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score,
    mean_absolute_error, mean_squared_error,r2_score,
    log_loss
)
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
# Utilities
combine = lambda *dicts: {k: v for d in dicts for (k, v) in d.items()}


class Timer():
    """Callable that maintains a list of timestamps.

    Attributes:
        times: A list of times the class instance was called, including
            initialization.
        total_time: Total accumulated time.
    """
    def __init__(self):
        self.times = [time.perf_counter()]
        self.total_time = 0.0

    def __call__(self, include_in_total: bool = True):
        self.times.append(time.perf_counter())
        delta_t = self.times[-1] - self.times[-2]
        if include_in_total:
            self.total_time += delta_t
        return delta_t


class LowCountGroupingTransformer(BaseEstimator):
    def __init__(self, min_obs=10, grouped_value='<OTHER>'):
        super().__init__()
        self.min_obs = min_obs
        self.grouped_value = grouped_value
        self.is_fitted = False
    
    def _fit_column(self, x):
        # THis is a pickling problem
        value_map = defaultdict(lambda: self.grouped_value)
        values = x.value_counts()
        for x in values.index[values >= self.min_obs]:
            value_map[x] = x
        return value_map
    
    def _transform_column(self, x, i):
        return x.map(self.value_maps[i], na_action='ignore')
    
    def fit(self, X, y=None):
        self.value_maps = [None] * X.shape[1]
        self.num_values = [None] * X.shape[1]
        for i, col in enumerate(X.columns):
            self.value_maps[i] = self._fit_column(X[col])
            self.num_values[i] = len(self.value_maps[i])
        self.is_fitted = True
        return self
    
    def transform(self, X):
        X = X.copy()
        for i, col in enumerate(X.columns):
            X.loc[:, col] = self._transform_column(X[col], i)
        return X
    
    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)


class LabelEncoder(BaseEstimator):
    def __init__(self):
        super().__init__()
        self.is_fitted = False
    
    def _fit_column(self, x):
        value_map = defaultdict(int)
        values = x.value_counts()
        for idx, x in enumerate(values.index):
            value_map[x] = idx
        return value_map
    
    def _transform_column(self, x, i):
        return x.map(self.value_maps[i], na_action='ignore').fillna(0)
    
    def fit(self, X, y=None):
        self.value_maps = [None] * X.shape[1]
        self.max_values = [None] * X.shape[1]
        X = X.copy()
        X = pd.DataFrame(X)
        for i, col in enumerate(X.columns):
            self.value_maps[i] = self._fit_column(X[col])
            self.max_values[i] = X[col].max()
        self.is_fitted = True
        return self
    
    def transform(self, X):
        X = X.copy()
        X = pd.DataFrame(X)
        for i, col in enumerate(X.columns):
            X.loc[:, col] = self._transform_column(X[col], i)
        return X
    
    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)


def calc_metrics_binary(probs, targets, threshold=0.5):
    pred_class = (probs > threshold).astype(np.int32)
    return {
        'n': len(probs),
        'auc': roc_auc_score(y_score=probs, y_true=targets),
        'accuracy': accuracy_score(y_pred=pred_class, y_true=targets),
        'precision': precision_score(y_pred=pred_class, y_true=targets),
        'recall': recall_score(y_pred=pred_class, y_true=targets),
        'f1': f1_score(y_pred=pred_class, y_true=targets),
        'log_loss': log_loss(y_pred=probs, y_true=targets)
    }

In [3]:
# Initialize data
data_in_base_path = '/kaggle/input/titanic'
splits = ('train', 'test')
data = {
    split: pd.read_csv(f'{data_in_base_path}/{split}.csv')
    for split in splits
}
test_cols = data['test'].columns
summaries = {
    'nunique': lambda x: x.nunique(),
    'dtype': lambda x: x.dtypes,
    'missing': lambda x: x.isnull().sum(),
}
pd.DataFrame({
    f'{smry}_{split}': summaries[smry](data[split][test_cols])
    for split in splits for smry in summaries.keys()
})

,nunique_train,dtype_train,missing_train,nunique_test,dtype_test,missing_test
PassengerId,891,int64,0,418,int64,0
Pclass,3,int64,0,3,int64,0
Name,891,object,0,418,object,0
Sex,2,object,0,2,object,0
Age,88,float64,177,79,float64,86
SibSp,7,int64,0,7,int64,0
Parch,7,int64,0,8,int64,0
Ticket,681,object,0,363,object,0
Fare,248,float64,0,169,float64,1
Cabin,147,object,687,76,object,327


In [4]:
# Create separate numeric and categorical versions of maybe-categorical columns
# Convert all categorical columns to strin
categorical_col_names = ['Pclass', 'Sex', 'Embarked']
maybe_cat_col_names = []  # 'Age', 'SibSp', 'Parch'
numeric_col_names = ['Fare', 'Age', 'SibSp', 'Parch']
unused_col_names = ['PassengerId', 'Name', 'Ticket', 'Cabin']
label_col_name = 'Survived'

for split in splits:
    for col_name in maybe_cat_col_names:
        data[split][col_name + '_cat'] = data[split][col_name].astype(str)
    for col_name in categorical_col_names:
        data[split][col_name] = data[split][col_name].astype(str)

numeric_col_names = numeric_col_names + maybe_cat_col_names
categorical_col_names = categorical_col_names + [x + '_cat' for x in maybe_cat_col_names]

In [5]:
# Drop unused columns and train/val split
x_train = data['train'][numeric_col_names + categorical_col_names]
y_train = data['train'][label_col_name]
x_test = data['test'][numeric_col_names + categorical_col_names]
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=23131)
x_train_full = data['train'][numeric_col_names + categorical_col_names]
y_train_full = data['train'][label_col_name]
print(x_train.shape)
print(x_train_full.shape)
display(x_train.head())
display(x_train_full.head())

(712, 7)
(891, 7)


,Fare,Age,SibSp,Parch,Pclass,Sex,Embarked
116,7.750,70.5,0,0,3,male,Q
387,13.000,36.0,0,0,2,female,S
729,7.925,25.0,1,0,3,female,S
347,16.100,NaN,1,0,3,female,S
848,33.000,28.0,0,1,2,male,S


,Fare,Age,SibSp,Parch,Pclass,Sex,Embarked
0,7.2500,22.0,1,0,3,male,S
1,71.2833,38.0,1,0,1,female,C
2,7.9250,26.0,0,0,3,female,S
3,53.1000,35.0,1,0,1,female,S
4,8.0500,35.0,0,0,3,male,S


In [6]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
label_encoder = Pipeline([
    ('grouper', LowCountGroupingTransformer(min_obs=10, grouped_value='<UNK>')),
    ('imputer', SimpleImputer(strategy='constant', fill_value='<UNK>')), 
    ('encoder', LabelEncoder()),
])
preprocessor_label_encoder = ColumnTransformer(transformers=[
    ('numeric', numeric_transformer, numeric_col_names),
    ('categorical', label_encoder, categorical_col_names),
])
models = {
#     'random_forest_entropy': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', RandomForestClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [3, 6, 10],
#             'classifier__criterion': ['entropy'],
#             'classifier__n_estimators': [500],
#         },
#         cv=3,
#     ),
#     'random_forest_gini': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', RandomForestClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [3, 6, 10],
#             'classifier__criterion': ['gini'],
#             'classifier__n_estimators': [500],
#         },
#         cv=3,
#     ),
#     'extra_trees_gini': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', ExtraTreesClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [6, 10, 15],
#             'classifier__criterion': ['gini'],
#             'classifier__n_estimators': [500],
#         },
#         cv=3,
#     ),
#     'extra_trees_entropy': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', ExtraTreesClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [6, 10, 15],
#             'classifier__criterion': ['entropy'],
#             'classifier__n_estimators': [500],
#         },
#         cv=3,
#     ),
#     'xgboost': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', XGBClassifier(verbosity=0, silent=True, use_label_encoder=False)),
#         ]),
#         param_grid={
#             'classifier__max_depth': [6, 10, 15],
#             'classifier__learning_rate': [0.1, 0.3, 0.5],
#         },
#         cv=3,
#     ),
#     'gbm': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', GradientBoostingClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [2, 3, 6, 10],
#         },
#         cv=3,
#     ),
#     'lightgbm': GridSearchCV(
#         Pipeline([
#             ('preprocessor', preprocessor_label_encoder),
#             ('classifier', LGBMClassifier()),
#         ]),
#         param_grid={
#             'classifier__max_depth': [-1, 3, 6, 10],
#             'classifier__learning_rate': [0.05, 0.1, 0.3],
#             'classifier__n_estimators': [500]
#         },
#         cv=5,
#     ),
}

In [7]:
# timer = Timer()
# for model_id, model in models.items():
#     models[model_id] = model.fit(x_train, y_train).best_estimator_
#     print(f'{model_id} fit in {timer():.3f} seconds')

In [8]:
# results = []
# for model_id, model in models.items():
#     print(model_id)
#     results.append(combine(
#         {'model_id': model_id}, 
#         calc_metrics_binary(model.predict_proba(x_val)[:, 1], y_val)
#     ))
#     print(model['classifier'].get_params())

# results = pd.DataFrame(results)
# display(results)

In [9]:
# catboost_params = CatBoostClassifier(
#     verbose=False, cat_features=categorical_col_names, 
# ).grid_search(
#     param_grid={
#         'learning_rate': [0.075, 0.1, 0.125],
#         'depth': [3, 4, 6],
#         'l2_leaf_reg': [3, 5, 7]
#     }, 
#     X=x_train, y=y_train, verbose=False
# )

# models['catboost'] = Pipeline([('classifier', CatBoostClassifier(
#     verbose=False, cat_features=categorical_col_names, 
#     **catboost_params['params']
# ))]).fit(x_train, y_train)

In [10]:
hyperparams = {
    'catboost': {'learning_rate': 0.1, 'depth': 4, 'l2_leaf_reg': 5, 'verbose': False, 'cat_features': ['Pclass', 'Sex', 'Embarked']},
    'random_forest_entropy': {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'entropy', 'max_depth': 6, 'max_features': 'auto', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_impurity_split': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 500, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False},
    'random_forest_gini': {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 6, 'max_features': 'auto', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_impurity_split': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 500, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False},
    'extra_trees_gini': {'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 6, 'max_features': 'auto', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_impurity_split': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 500, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False},
    'extra_trees_entropy': {'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'entropy', 'max_depth': 6, 'max_features': 'auto', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_impurity_split': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 500, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False},
    'xgboost': {'objective': 'binary:logistic', 'use_label_encoder': False, 'base_score': 0.5, 'booster': 'gbtree', 'colsample_bylevel': 1, 'colsample_bynode': 1, 'colsample_bytree': 1, 'enable_categorical': False, 'gamma': 0, 'gpu_id': -1, 'importance_type': None, 'interaction_constraints': '', 'learning_rate': 0.1, 'max_delta_step': 0, 'max_depth': 6, 'min_child_weight': 1, 'monotone_constraints': '()', 'n_estimators': 100, 'n_jobs': 4, 'num_parallel_tree': 1, 'predictor': 'auto', 'random_state': 0, 'reg_alpha': 0, 'reg_lambda': 1, 'scale_pos_weight': 1, 'subsample': 1, 'tree_method': 'exact', 'validate_parameters': 1, 'verbosity': 0, 'silent': True},
    'gbm': {'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'deviance', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_impurity_split': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'presort': 'deprecated', 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False},
    'lightgbm': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': 3, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': None, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'silent': 'warn', 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0},
}
models = {
    'catboost': Pipeline([
        ('classifier', CatBoostClassifier(**hyperparams['catboost'])),
    ]),
    'random_forest_entropy': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', RandomForestClassifier(**hyperparams['random_forest_entropy'])),
    ]),
    'random_forest_gini': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', RandomForestClassifier(**hyperparams['random_forest_gini'])),
    ]),
    'extra_trees_entropy': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', ExtraTreesClassifier(**hyperparams['extra_trees_entropy'])),
    ]),
    'extra_trees_gini': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', ExtraTreesClassifier(**hyperparams['extra_trees_gini'])),
    ]),
    'xgboost': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', XGBClassifier(**hyperparams['xgboost'])),
    ]),
    'gbm': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', GradientBoostingClassifier(**hyperparams['gbm'])),
    ]),
    'lightgbm': Pipeline([
        ('preprocessor', preprocessor_label_encoder),
        ('classifier', LGBMClassifier(**hyperparams['lightgbm'])),
    ]),
}

In [11]:
timer = Timer()
results = []
for model_id, model in models.items():
    models[model_id] = model.fit(x_train, y_train)
    print(f'{model_id} fit in {timer():.3f} seconds')
    results.append(combine(
        {'model_id': model_id}, 
        calc_metrics_binary(model.predict_proba(x_val)[:, 1], y_val)
    ))

results = pd.DataFrame(results)
display(results)

catboost fit in 1.323 seconds
random_forest_entropy fit in 1.019 seconds
random_forest_gini fit in 1.227 seconds
extra_trees_entropy fit in 0.768 seconds
extra_trees_gini fit in 0.763 seconds
xgboost fit in 0.520 seconds
gbm fit in 0.156 seconds
lightgbm fit in 0.165 seconds


,model_id,n,auc,accuracy,precision,recall,f1,log_loss
0,catboost,179,0.846835,0.787709,0.805970,0.683544,0.739726,0.530412
1,random_forest_entropy,179,0.866392,0.787709,0.901961,0.582278,0.707692,0.469185
2,random_forest_gini,179,0.867025,0.787709,0.901961,0.582278,0.707692,0.469337
3,extra_trees_entropy,179,0.845443,0.770950,0.851852,0.582278,0.691729,0.499440
4,extra_trees_gini,179,0.853924,0.776536,0.867925,0.582278,0.696970,0.496625
5,xgboost,179,0.874684,0.787709,0.836066,0.645570,0.728571,0.455160
6,gbm,179,0.858228,0.804469,0.892857,0.632911,0.740741,0.473647
7,lightgbm,179,0.876646,0.804469,0.843750,0.683544,0.755245,0.467124


In [12]:
model_stacked = StackingClassifier(
    estimators=list(models.items()), 
    final_estimator=LogisticRegressionCV(Cs=50), 
    cv=3
).fit(x_train, y_train)
print(model_stacked.final_estimator_.coef_)
print(model_stacked.final_estimator_.intercept_)
print(model_stacked.final_estimator_.C_)

[[0.32422145 0.29583259 0.29529442 0.27713091 0.27935233 0.31877052
  0.32638026 0.32023132]]
[-1.49491834]
[0.00910298]


In [13]:
models['stacked'] = model_stacked
results = pd.DataFrame([
    combine(
        {'model_id': model_id, 'split': 'val'}, 
        calc_metrics_binary(model.predict_proba(x_val)[:, 1], y_val)
    )
    for model_id, model in models.items()
])
display(results)

results = pd.DataFrame([
    combine(
        {'model_id': model_id, 'split': 'train'}, 
        calc_metrics_binary(model.predict_proba(x_train)[:, 1], y_train)
    )
    for model_id, model in models.items()
])
display(results)

# with open('model.pickle', 'wb') as fout:
#     pickle.dump(model_stacked_full, fout)

,model_id,split,n,auc,accuracy,precision,recall,f1,log_loss
0,catboost,val,179,0.846835,0.787709,0.805970,0.683544,0.739726,0.530412
1,random_forest_entropy,val,179,0.866392,0.787709,0.901961,0.582278,0.707692,0.469185
2,random_forest_gini,val,179,0.867025,0.787709,0.901961,0.582278,0.707692,0.469337
3,extra_trees_entropy,val,179,0.845443,0.770950,0.851852,0.582278,0.691729,0.499440
4,extra_trees_gini,val,179,0.853924,0.776536,0.867925,0.582278,0.696970,0.496625
5,xgboost,val,179,0.874684,0.787709,0.836066,0.645570,0.728571,0.455160
6,gbm,val,179,0.858228,0.804469,0.892857,0.632911,0.740741,0.473647
7,lightgbm,val,179,0.876646,0.804469,0.843750,0.683544,0.755245,0.467124
8,stacked,val,179,0.875443,0.776536,0.933333,0.531646,0.677419,0.523234


,model_id,split,n,auc,accuracy,precision,recall,f1,log_loss
0,catboost,train,712,0.968807,0.918539,0.875458,0.908745,0.891791,0.222500
1,random_forest_entropy,train,712,0.934129,0.869382,0.912621,0.714829,0.801706,0.329905
2,random_forest_gini,train,712,0.935476,0.875000,0.922330,0.722433,0.810235,0.325301
3,extra_trees_entropy,train,712,0.904596,0.852528,0.852679,0.726236,0.784394,0.379664
4,extra_trees_gini,train,712,0.904871,0.849719,0.845133,0.726236,0.781186,0.379240
5,xgboost,train,712,0.976788,0.926966,0.956710,0.840304,0.894737,0.210141
6,gbm,train,712,0.953268,0.904494,0.918455,0.813688,0.862903,0.270754
7,lightgbm,train,712,0.959005,0.907303,0.922747,0.817490,0.866935,0.252843
8,stacked,train,712,0.958577,0.884831,0.959391,0.718631,0.821739,0.414367


In [14]:
# from sklearn.calibration import CalibratedClassifierCV
# model_stacked_calibrated = CalibratedClassifierCV(
#     base_estimator=model_stacked, 
#     cv=3
# ).fit(x_train, y_train)

In [15]:
model_stacked_full = model_stacked.fit(x_train_full, y_train_full)
print(model_stacked_full.final_estimator_.coef_)
print(model_stacked_full.final_estimator_.intercept_)
print(model_stacked_full.final_estimator_.C_)

output = pd.DataFrame({
    'PassengerId': data['test']['PassengerId'], 
    'Survived': model_stacked_full.predict(x_test)
})

output.to_csv('submission5.csv', index=False)
display(output)

[[0.56716841 0.41094259 0.41408701 0.38685649 0.38265504 0.47487524
  0.46046027 0.4894276 ]]
[-1.89632189]
[0.01930698]


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [16]:
print(data['train']['Survived'].value_counts())
print(output['Survived'].value_counts())

0    549
1    342
Name: Survived, dtype: int64
0    292
1    126
Name: Survived, dtype: int64
